# Notebook 01 - Preprocessing Pipeline

## Goal
Build a standard pipeline: resample, normalize, pre-emphasis, and VAD.


## Agenda
- Resample to 16kHz
- Peak normalize
- Apply pre-emphasis
- Trim low-energy frames


## Concept and Math

Pre-emphasis: y[n] = x[n] - alpha x[n-1], often alpha = 0.97.
It boosts high-frequency energy reduced by speech production and channels.
Energy-based VAD drops low-RMS frames as likely silence/non-speech.


In [ ]:
from pathlib import Path
import numpy as np
import librosa as lb
import librosa.display
import matplotlib.pyplot as plt

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))
if not audio_files:
    raise FileNotFoundError("No .flac or .wav found under ../dataset")

audio_path = audio_files[0]
print(f"Using: {audio_path}")

wave, sr = lb.load(audio_path, sr=None, mono=True)
if sr != 16000:
    wave = lb.resample(wave, orig_sr=sr, target_sr=16000)
    sr = 16000

wave = wave / (np.max(np.abs(wave)) + 1e-8)
alpha = 0.97
wave_pre = np.append(wave[0], wave[1:] - alpha * wave[:-1])

frame_len = int(0.025 * sr)
hop_len = int(0.010 * sr)
rms = lb.feature.rms(y=wave_pre, frame_length=frame_len, hop_length=hop_len)[0]
threshold = np.percentile(rms, 20)
print("rms threshold:", threshold)


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
import torch
import torchaudio

wave_t, sr_t = torchaudio.load(str(audio_path))
if wave_t.shape[0] > 1:
    wave_t = wave_t.mean(dim=0, keepdim=True)
if sr_t != 16000:
    wave_t = torchaudio.functional.resample(wave_t, sr_t, 16000)
wave_t = wave_t / (wave_t.abs().max() + 1e-8)
pre_t = torch.cat([wave_t[:, :1], wave_t[:, 1:] - 0.97 * wave_t[:, :-1]], dim=1)
print(pre_t.shape)


## Review Checklist
- Why 16kHz for anti-spoofing pipelines?
- What does pre-emphasis change in spectrum?
- What are limitations of energy-only VAD?
